# Projeto Semantix — Análise de Retenção de Clientes em E-commerce

Este notebook apresenta o processo de preparação, limpeza e análise inicial de um dataset de comportamento de clientes em uma plataforma global de e-commerce. O objetivo do projeto é investigar padrões associados ao abandono de clientes e preparar uma base tratada para criação de visualizações no Looker Studio.


## 1. Configuração do Ambiente

Nesta etapa são instaladas e importadas as bibliotecas necessárias para utilização do PySpark no Google Colab.


In [25]:
!pip install pyspark


In [26]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

In [27]:
import google.colab
google.colab.drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
spark = SparkSession.builder.getOrCreate()

## 2. Coleta dos Dados

Os dados foram obtidos a partir do dataset público **Ecommerce Customer Behavior Dataset**, disponível no Kaggle. O arquivo foi armazenado no Google Drive e carregado no ambiente do Google Colab para tratamento com PySpark.


In [29]:
df = spark.read.csv('/content/drive/MyDrive/Curso EBAC/Projeto/ecommerce_customer_churn_dataset.csv', header=True, inferSchema=True)

df.show(5)
df.printSchema()

+----+------+-------+----------+----------------+---------------+--------------------+-----------------+---------------------+--------------+---------------+-------------------+------------------------+-------------------+------------+---------------+----------------------+-----------------------+-----------------------------+----------------+------------------------+--------------+--------------+-------+--------------+
| Age|Gender|Country|      City|Membership_Years|Login_Frequency|Session_Duration_Avg|Pages_Per_Session|Cart_Abandonment_Rate|Wishlist_Items|Total_Purchases|Average_Order_Value|Days_Since_Last_Purchase|Discount_Usage_Rate|Returns_Rate|Email_Open_Rate|Customer_Service_Calls|Product_Reviews_Written|Social_Media_Engagement_Score|Mobile_App_Usage|Payment_Method_Diversity|Lifetime_Value|Credit_Balance|Churned|Signup_Quarter|
+----+------+-------+----------+----------------+---------------+--------------------+-----------------+---------------------+--------------+-----------

## 3. Análise Inicial dos Dados

Nesta etapa é feita uma primeira inspeção da base, incluindo quantidade de linhas e colunas, criação de uma tabela temporária para consultas SQL, identificação de valores nulos e verificação de registros duplicados.


In [30]:
# Verificando total de linhas e colunas
total_linhas = df.count()
total_colunas = len(df.columns)

print(f"Total de linhas: {total_linhas}")
print(f"Total de colunas: {total_colunas}")

Total de linhas: 50000
Total de colunas: 25


In [31]:
# Criando uma tabela temporaria "ecommerce_temp"
df.createOrReplaceTempView("ecommerce_temp")
spark.sql("""
SELECT *
FROM ecommerce_temp
LIMIT 5
""").show()

+----+------+-------+----------+----------------+---------------+--------------------+-----------------+---------------------+--------------+---------------+-------------------+------------------------+-------------------+------------+---------------+----------------------+-----------------------+-----------------------------+----------------+------------------------+--------------+--------------+-------+--------------+
| Age|Gender|Country|      City|Membership_Years|Login_Frequency|Session_Duration_Avg|Pages_Per_Session|Cart_Abandonment_Rate|Wishlist_Items|Total_Purchases|Average_Order_Value|Days_Since_Last_Purchase|Discount_Usage_Rate|Returns_Rate|Email_Open_Rate|Customer_Service_Calls|Product_Reviews_Written|Social_Media_Engagement_Score|Mobile_App_Usage|Payment_Method_Diversity|Lifetime_Value|Credit_Balance|Churned|Signup_Quarter|
+----+------+-------+----------+----------------+---------------+--------------------+-----------------+---------------------+--------------+-----------

In [32]:
# Verificando quantidade de valores nulos em cada coluna
for coluna in df.columns:
    total_nulos = df.filter(col(coluna).isNull()).count()
    print(f"{coluna}: {total_nulos}")

Age: 2495
Gender: 0
Country: 0
City: 0
Membership_Years: 0
Login_Frequency: 0
Session_Duration_Avg: 3399
Pages_Per_Session: 3000
Cart_Abandonment_Rate: 0
Wishlist_Items: 4000
Total_Purchases: 0
Average_Order_Value: 0
Days_Since_Last_Purchase: 3000
Discount_Usage_Rate: 3500
Returns_Rate: 4491
Email_Open_Rate: 2528
Customer_Service_Calls: 168
Product_Reviews_Written: 3500
Social_Media_Engagement_Score: 6000
Mobile_App_Usage: 5000
Payment_Method_Diversity: 2500
Lifetime_Value: 0
Credit_Balance: 5500
Churned: 0
Signup_Quarter: 0


In [33]:
# Verificando valores duplicados
total_original = df.count()
total_sem_duplicados = df.dropDuplicates().count()

print(f"Total original: {total_original}")
print(f"Total sem duplicados: {total_sem_duplicados}")
print(f"Duplicados encontrados: {total_original - total_sem_duplicados}")

Total original: 50000
Total sem duplicados: 50000
Duplicados encontrados: 0


## 4. Limpeza e Tratamento dos Dados

Foram realizadas padronizações nos nomes das colunas, conversão inicial dos campos para tipos adequados, remoção de espaços em branco e padronização de textos categóricos com `TRIM` e `INITCAP`.


In [34]:
# Tratando dados e limpando, nome de colunas para facilitar consultas
df_limpo = spark.sql("""
SELECT
    CAST(Age AS INT) AS idade,
    INITCAP(TRIM(Gender)) AS genero,
    INITCAP(TRIM(Country)) AS pais,
    INITCAP(TRIM(City)) AS cidade,
    CAST(Membership_Years AS DOUBLE) AS anos_assinatura,
    CAST(Login_Frequency AS INT) AS frequencia_login,
    CAST(Session_Duration_Avg AS DOUBLE) AS duracao_media_sessao,
    CAST(Pages_Per_Session AS DOUBLE) AS paginas_por_sessao,
    CAST(Cart_Abandonment_Rate AS DOUBLE) AS taxa_abandono_carrinho,
    CAST(Wishlist_Items AS INT) AS itens_lista_desejos,
    CAST(Total_Purchases AS INT) AS total_compras,
    CAST(Average_Order_Value AS DOUBLE) AS valor_medio_pedido,
    CAST(Days_Since_Last_Purchase AS INT) AS dias_desde_ultima_compra,
    CAST(Discount_Usage_Rate AS DOUBLE) AS taxa_uso_desconto,
    CAST(Returns_Rate AS DOUBLE) AS taxa_devolucao,
    CAST(Email_Open_Rate AS DOUBLE) AS taxa_abertura_email,
    CAST(Customer_Service_Calls AS INT) AS chamadas_suporte,
    CAST(Product_Reviews_Written AS INT) AS avaliacoes_escritas,
    CAST(Social_Media_Engagement_Score AS DOUBLE) AS engajamento_social,
    CAST(Mobile_App_Usage AS DOUBLE) AS uso_app,
    CAST(Payment_Method_Diversity AS INT) AS diversidade_pagamento,
    CAST(Lifetime_Value AS DOUBLE) AS valor_cliente,
    CAST(Credit_Balance AS DOUBLE) AS saldo_credito,
    CAST(Churned AS INT) AS churn,
    Signup_Quarter AS trimestre_cadastro

FROM ecommerce_temp
""")

In [35]:
df_limpo.createOrReplaceTempView("ecommerce_limpo")
df_limpo.show(5)

+-----+------+------+----------+---------------+----------------+--------------------+------------------+----------------------+-------------------+-------------+------------------+------------------------+------------------+--------------+-------------------+----------------+-------------------+------------------+-------+---------------------+-------------+-------------+-----+------------------+
|idade|genero|  pais|    cidade|anos_assinatura|frequencia_login|duracao_media_sessao|paginas_por_sessao|taxa_abandono_carrinho|itens_lista_desejos|total_compras|valor_medio_pedido|dias_desde_ultima_compra| taxa_uso_desconto|taxa_devolucao|taxa_abertura_email|chamadas_suporte|avaliacoes_escritas|engajamento_social|uso_app|diversidade_pagamento|valor_cliente|saldo_credito|churn|trimestre_cadastro|
+-----+------+------+----------+---------------+----------------+--------------------+------------------+----------------------+-------------------+-------------+------------------+-------------------

## 5. Tratamento de Valores Ausentes

O dataset apresenta valores ausentes em algumas colunas. Para evitar a perda de muitos registros, os campos numéricos com nulos foram preenchidos pela média da própria coluna.


In [36]:
# verificando se ainda existem nulos
from pyspark.sql.functions import col, count, when

for coluna in df_limpo.columns:
    total_nulos = df_limpo.filter(col(coluna).isNull()).count()
    print(f"{coluna}: {total_nulos}")

idade: 2495
genero: 0
pais: 0
cidade: 0
anos_assinatura: 0
frequencia_login: 0
duracao_media_sessao: 3399
paginas_por_sessao: 3000
taxa_abandono_carrinho: 0
itens_lista_desejos: 4000
total_compras: 0
valor_medio_pedido: 0
dias_desde_ultima_compra: 3000
taxa_uso_desconto: 3500
taxa_devolucao: 4491
taxa_abertura_email: 2528
chamadas_suporte: 168
avaliacoes_escritas: 3500
engajamento_social: 6000
uso_app: 5000
diversidade_pagamento: 2500
valor_cliente: 0
saldo_credito: 5500
churn: 0
trimestre_cadastro: 0


In [37]:
# Substituindo nulos pela média
from pyspark.sql.functions import avg

media_idade = df_limpo.select(avg("idade")).first()[0]

In [38]:
from pyspark.sql.functions import mean

colunas_numericas = [
    'idade',
    'duracao_media_sessao',
    'paginas_por_sessao',
    'itens_lista_desejos',
    'dias_desde_ultima_compra',
    'taxa_uso_desconto',
    'taxa_devolucao',
    'taxa_abertura_email',
    'chamadas_suporte',
    'avaliacoes_escritas',
    'engajamento_social',
    'uso_app',
    'diversidade_pagamento',
    'saldo_credito'
]

valores_preenchimento = {}

for coluna in colunas_numericas:
    media = df_limpo.select(mean(coluna)).first()[0]
    valores_preenchimento[coluna] = media

In [39]:
df_limpo = df_limpo.fillna(valores_preenchimento)

for coluna in df_limpo.columns:
    total_nulos = df_limpo.filter(col(coluna).isNull()).count()
    print(f"{coluna}: {total_nulos}")

idade: 0
genero: 0
pais: 0
cidade: 0
anos_assinatura: 0
frequencia_login: 0
duracao_media_sessao: 0
paginas_por_sessao: 0
taxa_abandono_carrinho: 0
itens_lista_desejos: 0
total_compras: 0
valor_medio_pedido: 0
dias_desde_ultima_compra: 0
taxa_uso_desconto: 0
taxa_devolucao: 0
taxa_abertura_email: 0
chamadas_suporte: 0
avaliacoes_escritas: 0
engajamento_social: 0
uso_app: 0
diversidade_pagamento: 0
valor_cliente: 0
saldo_credito: 0
churn: 0
trimestre_cadastro: 0


## 6. Tratamento de Outliers

Foi realizada uma validação da coluna de idade para identificar valores fora de um intervalo plausível para clientes de e-commerce. Registros com idade inferior a 18 anos ou superior a 100 anos foram considerados inconsistentes e removidos da análise.


In [40]:
# Verificando Idade
spark.sql("""
SELECT
MIN(idade) AS idade_minima,
MAX(idade) AS idade_maxima,
AVG(idade) AS idade_media
FROM ecommerce_limpo
""").show()

+------------+------------+-----------------+
|idade_minima|idade_maxima|      idade_media|
+------------+------------+-----------------+
|           5|         200|37.80296810862014|
+------------+------------+-----------------+



In [41]:
# Vou trabalhar com um intervalo de idade entre 18 e 100 anos
spark.sql("""
SELECT COUNT(*) AS total_idades_invalidas
FROM ecommerce_limpo
WHERE idade < 18
   OR idade > 100
""").show()

+----------------------+
|total_idades_invalidas|
+----------------------+
|                    50|
+----------------------+



In [42]:
# Removendo essas idades invalidas
df_limpo = spark.sql("""
SELECT *
FROM ecommerce_limpo
WHERE idade BETWEEN 18 AND 100
""")

In [43]:
df_limpo.createOrReplaceTempView("ecommerce_limpo")

spark.sql("""
SELECT
MIN(idade) AS idade_minima,
MAX(idade) AS idade_maxima,
AVG(idade) AS idade_media
FROM ecommerce_limpo
""").show()

+------------+------------+------------------+
|idade_minima|idade_maxima|       idade_media|
+------------+------------+------------------+
|          18|          75|37.763354757138345|
+------------+------------+------------------+



Foram identificados registros com idades inconsistentes, incluindo valores inferiores a 18 anos e superiores a 100 anos. Esses registros foram considerados outliers e removidos da análise para garantir maior confiabilidade dos resultados.

## 7. Validação da Variável de Abandono

A variável `churn` indica a situação do cliente: `0` para cliente ativo e `1` para cliente perdido. Essa validação permite calcular a taxa de abandono da base.


In [44]:
# Churn
spark.sql("""
SELECT
churn,
COUNT(*) AS total
FROM ecommerce_limpo
GROUP BY churn
ORDER BY churn
""").show()

+-----+-----+
|churn|total|
+-----+-----+
|    0|33657|
|    1|13798|
+-----+-----+



## Taxa de Abandono de Clientes

Clientes perdidos: **13.798**  
Total de clientes: **47.455**  

Taxa de Abandono = (13.798 / 47.455) × 100 = **29,08%**

Neste projeto, o termo **abandono de clientes** é utilizado como uma forma mais clara de representar o conceito técnico de *churn*.


## 8. Criação das Variáveis de Negócio

Foram criadas variáveis derivadas para facilitar a análise e a construção do dashboard no Looker Studio, como faixa etária, status do cliente e nível de risco de abandono.


In [45]:
# criar as variáveis de negócio que vão alimentar o dashboard.
df_final = spark.sql("""
SELECT
    *,

    CASE
        WHEN idade BETWEEN 18 AND 24 THEN '18-24'
        WHEN idade BETWEEN 25 AND 34 THEN '25-34'
        WHEN idade BETWEEN 35 AND 44 THEN '35-44'
        WHEN idade BETWEEN 45 AND 54 THEN '45-54'
        ELSE '55+'
    END AS faixa_etaria,

    CASE
        WHEN churn = 1 THEN 'Cliente Perdido'
        ELSE 'Cliente Ativo'
    END AS status_cliente,

    CASE
        WHEN dias_desde_ultima_compra <= 30 THEN 'Baixo Risco'
        WHEN dias_desde_ultima_compra BETWEEN 31 AND 90 THEN 'Risco Médio'
        ELSE 'Alto Risco'
    END AS risco_churn

FROM ecommerce_limpo
""")

df_final.createOrReplaceTempView("ecommerce_final")

## 9. Padronização dos Tipos de Dados

Durante a preparação da base, algumas colunas numéricas poderiam ser interpretadas incorretamente como texto em ferramentas de visualização. Para garantir análises corretas, agregações estatísticas e criação de métricas no Looker Studio, os campos foram convertidos explicitamente para os tipos `integer` e `double`.


In [46]:
from pyspark.sql.functions import col

df_final = df_final \
    .withColumn("idade", col("idade").cast("integer")) \
    .withColumn("anos_assinatura", col("anos_assinatura").cast("double")) \
    .withColumn("frequencia_login", col("frequencia_login").cast("integer")) \
    .withColumn("duracao_media_sessao", col("duracao_media_sessao").cast("double")) \
    .withColumn("paginas_por_sessao", col("paginas_por_sessao").cast("double")) \
    .withColumn("taxa_abandono_carrinho", col("taxa_abandono_carrinho").cast("double")) \
    .withColumn("itens_lista_desejos", col("itens_lista_desejos").cast("integer")) \
    .withColumn("total_compras", col("total_compras").cast("integer")) \
    .withColumn("valor_medio_pedido", col("valor_medio_pedido").cast("double")) \
    .withColumn("dias_desde_ultima_compra", col("dias_desde_ultima_compra").cast("integer")) \
    .withColumn("taxa_uso_desconto", col("taxa_uso_desconto").cast("double")) \
    .withColumn("taxa_devolucao", col("taxa_devolucao").cast("double")) \
    .withColumn("taxa_abertura_email", col("taxa_abertura_email").cast("double")) \
    .withColumn("chamadas_suporte", col("chamadas_suporte").cast("integer")) \
    .withColumn("avaliacoes_escritas", col("avaliacoes_escritas").cast("integer")) \
    .withColumn("engajamento_social", col("engajamento_social").cast("double")) \
    .withColumn("uso_app", col("uso_app").cast("double")) \
    .withColumn("diversidade_pagamento", col("diversidade_pagamento").cast("integer")) \
    .withColumn("valor_cliente", col("valor_cliente").cast("double")) \
    .withColumn("saldo_credito", col("saldo_credito").cast("double")) \
    .withColumn("churn", col("churn").cast("integer"))

df_final.printSchema()

root
 |-- idade: integer (nullable = true)
 |-- genero: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- anos_assinatura: double (nullable = true)
 |-- frequencia_login: integer (nullable = true)
 |-- duracao_media_sessao: double (nullable = true)
 |-- paginas_por_sessao: double (nullable = true)
 |-- taxa_abandono_carrinho: double (nullable = true)
 |-- itens_lista_desejos: integer (nullable = true)
 |-- total_compras: integer (nullable = true)
 |-- valor_medio_pedido: double (nullable = true)
 |-- dias_desde_ultima_compra: integer (nullable = true)
 |-- taxa_uso_desconto: double (nullable = true)
 |-- taxa_devolucao: double (nullable = true)
 |-- taxa_abertura_email: double (nullable = true)
 |-- chamadas_suporte: integer (nullable = true)
 |-- avaliacoes_escritas: integer (nullable = true)
 |-- engajamento_social: double (nullable = true)
 |-- uso_app: double (nullable = true)
 |-- diversidade_pagamento: integer (nullable = true

In [ ]:
# Atualizando a tabela temporária após a padronização dos tipos de dados

df_final.createOrReplaceTempView("ecommerce_final")


## 10. Análise Exploratória dos Dados (EDA)

A análise exploratória tem como objetivo identificar padrões de comportamento associados ao abandono de clientes, incluindo distribuição de clientes ativos e perdidos, taxa de abandono por faixa etária, comportamento de uso do aplicativo, valor do cliente e tempo desde a última compra.


### 10.1 KPIs Principais

Foram calculados os principais indicadores da base: total de clientes, taxa de abandono e distribuição entre clientes ativos e perdidos.


In [51]:
# Total de clientes

df_final.createOrReplaceTempView("ecommerce_final")

spark.sql("""
SELECT COUNT(*) AS total_clientes
FROM ecommerce_final
""").show()

# Taxa de Churn

spark.sql("""
SELECT
    ROUND(
        SUM(churn) * 100.0 / COUNT(*),
        2
    ) AS taxa_churn
FROM ecommerce_final
""").show()



+--------------+
|total_clientes|
+--------------+
|         47455|
+--------------+

+----------+
|taxa_churn|
+----------+
|     29.08|
+----------+



In [52]:
# Clientes ativos x perdidos

spark.sql("""
SELECT
    status_cliente,
    COUNT(*) AS total
FROM ecommerce_final
GROUP BY status_cliente
""").show()

+---------------+-----+
| status_cliente|total|
+---------------+-----+
|Cliente Perdido|13798|
|  Cliente Ativo|33657|
+---------------+-----+



### 10.2 Primeiros Insights

As consultas abaixo servem como apoio para a interpretação dos dados e para a construção das visualizações no Looker Studio.


In [46]:

# Churn por faixa etária , com objetivo de descobrir qual faixa etaria abandona mais a plataforma

spark.sql("""
SELECT
    faixa_etaria,
    ROUND(AVG(churn)*100,2) AS taxa_churn
FROM ecommerce_final
GROUP BY faixa_etaria
ORDER BY faixa_etaria
""").show()

+------------+----------+
|faixa_etaria|taxa_churn|
+------------+----------+
|       18-24|     48.38|
|       25-34|      26.1|
|       35-44|     25.54|
|       45-54|      26.1|
|         55+|     26.58|
+------------+----------+



In [47]:
# Churn por país, existem países com risco maior de abandono?

spark.sql("""
SELECT
    pais,
    ROUND(AVG(churn)*100,2) AS taxa_churn
FROM ecommerce_final
GROUP BY pais
ORDER BY taxa_churn DESC
LIMIT 10
""").show()

+---------+----------+
|     pais|taxa_churn|
+---------+----------+
|Australia|     30.07|
|   Canada|     29.37|
|    India|     29.36|
|  Germany|     29.22|
|      Usa|     29.14|
|       Uk|     29.13|
|    Japan|     27.92|
|   France|     27.55|
+---------+----------+



In [48]:
# Uso do app x churn , Clientes que usam mais o aplicativo permanecem mais tempo?

spark.sql("""
SELECT
    status_cliente,
    ROUND(AVG(uso_app),2) AS media_uso_app
FROM ecommerce_final
GROUP BY status_cliente
""").show()

+---------------+-------------+
| status_cliente|media_uso_app|
+---------------+-------------+
|Cliente Perdido|        16.13|
|  Cliente Ativo|        20.72|
+---------------+-------------+



In [49]:
# Lifetime Value x churn , Clientes com maior valor permanecem mais na plataforma?

spark.sql("""
SELECT
    status_cliente,
    ROUND(AVG(valor_cliente),2) AS valor_medio_cliente
FROM ecommerce_final
GROUP BY status_cliente
""").show()

+---------------+-------------------+
| status_cliente|valor_medio_cliente|
+---------------+-------------------+
|Cliente Perdido|            1426.22|
|  Cliente Ativo|            1446.86|
+---------------+-------------------+



In [50]:
# Dias sem comprar x churn, Quanto mais tempo sem comprar, maior o risco de churn?

spark.sql("""
SELECT
    status_cliente,
    ROUND(AVG(dias_desde_ultima_compra),2) AS media_dias
FROM ecommerce_final
GROUP BY status_cliente
""").show()

+---------------+----------+
| status_cliente|media_dias|
+---------------+----------+
|Cliente Perdido|     36.92|
|  Cliente Ativo|     26.91|
+---------------+----------+



## 11. Exportação dos Dados para o Looker Studio

Após a limpeza, tratamento, criação das variáveis de negócio e validações iniciais, a base final foi exportada em formato CSV para ser utilizada no Looker Studio.


In [53]:
df_final.coalesce(1)\
.write\
.mode("overwrite")\
.option("header", True)\
.csv("/content/drive/MyDrive/Curso EBAC/Projeto/ecommerce_dashboard")

In [ ]:
# Opcional: copiar o arquivo CSV gerado pelo Spark para um nome mais amigável
# O Spark salva o resultado dentro de uma pasta com o nome part-0000...

import os
import shutil

pasta_saida = "/content/drive/MyDrive/Curso EBAC/Projeto/ecommerce_dashboard"
arquivo_final = "/content/drive/MyDrive/Curso EBAC/Projeto/ecommerce_dashboard.csv"

for arquivo in os.listdir(pasta_saida):
    if arquivo.endswith(".csv"):
        shutil.copy(os.path.join(pasta_saida, arquivo), arquivo_final)
        print(f"Arquivo CSV final salvo em: {arquivo_final}")


## 12. Considerações Finais

A base foi tratada e preparada para análise visual no Looker Studio. O processo incluiu tratamento de valores nulos, padronização de tipos de dados, remoção de outliers e criação de variáveis analíticas. Com isso, foi possível construir um dashboard voltado à análise de retenção, comportamento e risco de abandono de clientes em uma plataforma global de e-commerce.
